# RODA · Predicción de ventas e inventario
## Cinco modelos, validación temporal y exportación para el dashboard

**Pregunta:** al cerrar hoy, ¿cuántas unidades pagadas de cada SKU se venderán en los próximos siete días?

952 productos, 730 días, 694,960 observaciones diarias y 622,608 ejemplos de aprendizaje. El historial es **simulado**: este experimento no demuestra precisión en ventas reales.

**Uso:** Colab con CPU → **Entorno de ejecución → Ejecutar todas**. No necesita GPU ni credenciales. El modo completo realiza 30 ajustes de búsqueda temporal y cinco ajustes finales; su duración depende de la CPU. Descarga el ZIP al terminar: incluye modelo, métricas, gráficos y contrato de inferencia.

El dashboard podrá mostrar qué modelos se probaron, cuál ganó y por qué. Los resultados se generan durante la ejecución; el ganador no está predeterminado.

**Evaluación completa guardada:** [tablas, matrices y 12 gráficos de la ejecución del 19/10/2026](https://github.com/NEYEN111/REPUESTOS/tree/main/ml/evaluations/roda-v1). Son resultados locales con datos simulados; ejecutar este notebook genera una nueva evaluación y su propio ZIP.

## 1. Código y entorno reproducible

Las versiones se fijan para volver a cargar el modelo con el mismo entorno. Se registra el commit y se comprueba el dataset mediante SHA-256. Conserva el ZIP completo de cada ejecución.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
IN_COLAB = "google.colab" in sys.modules
REPOSITORY = "https://github.com/NEYEN111/REPUESTOS.git"
SOURCE_REF = "2e646cca37afb457e3c68840a6ef8106e1a807b0"  # Código validado; reproducible aunque cambie main.
if IN_COLAB:
    REPO = Path("/content/roda-ml")
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    subprocess.run(["git", "fetch", "origin", SOURCE_REF], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "ml/requirements.txt")], check=True)
    from importlib.metadata import version as installed_version
    for module, package in [("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn")]:
        loaded = sys.modules.get(module)
        if loaded is not None and getattr(loaded, "__version__", installed_version(package)) != installed_version(package):
            raise RuntimeError("Las versiones cargadas cambiaron. Reinicia la sesión de Colab y ejecuta todas las celdas otra vez.")
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml/config.json").exists())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
print("Python:", sys.version.split()[0])

## 2. Configuración y motivos para comparar cada modelo

Mantén el modo completo para la exposición. El modo técnico utiliza 48 SKU y búsqueda reducida: valida funcionamiento, **no sustituye la evaluación del catálogo**.

Los cinco candidatos son Ridge, PoissonRegressor, RandomForestRegressor, ExtraTreesRegressor e HistGradientBoostingRegressor. Se contrastan además dos baselines sin entrenamiento: última semana y promedio de 28 días.

In [ ]:
from datetime import datetime
from importlib.metadata import version
import pandas as pd
from IPython.display import display, Markdown, Image
from ml.train import load_dataset, run_training, MODEL_REASONS

SMOKE = os.environ.get("RODA_ML_SMOKE", "0") == "1"  # False por defecto en Colab.
OUTPUT = REPO / "ml/runs" / (datetime.now().strftime("%Y%m%d-%H%M%S") + ("-tecnico" if SMOKE else "-completo"))
DATA = REPO / "data/ml/roda-v1"
config = json.loads((REPO / "ml/config.json").read_text(encoding="utf-8"))
print("Modo:", "PRUEBA TÉCNICA" if SMOKE else "COMPLETO")
display(pd.DataFrame([{"Librería": p, "Versión": version(p)} for p in
                     ["numpy", "pandas", "scipy", "scikit-learn", "joblib", "matplotlib"]]))
display(pd.DataFrame([{"Modelo": k, "Por qué compararlo": v} for k, v in MODEL_REASONS.items()]))
display(pd.DataFrame([{"Modelo": k, "Configuraciones": str(v)} for k, v in config["models"].items()]))

## 3. Integridad, objetivo y variables

Una fila es un **SKU en una fecha de corte**. Un kit o juego cuenta como una unidad del catálogo. El objetivo suma pagos confirmados de t+1 a t+7; no descuenta devoluciones ni representa demanda sin restricciones. Los importes del dataset están en céntimos.

Las variables usan información disponible hasta el cierre de t: ventas, lags, visitas, carritos, exposición al stock, reservas, compras pendientes, precio y calendario conocido. Los días desconocidos no son ventas cero.

El campo diagnostico_quiebre_futuro_7d contiene información futura: sólo se usa al evaluar errores. Se excluye expresamente de los predictores junto con objetivos, fechas de etiqueta y particiones.

In [ ]:
data, manifest = load_dataset(DATA, smoke=SMOKE)
display(pd.DataFrame({"Control": ["Filas", "SKU", "Features", "Nulos", "Duplicados fecha/SKU"],
                      "Resultado": [len(data), data.sku.nunique(), len(manifest["feature_columns"]),
                                    int(data.isna().sum().sum()), int(data.duplicated(["sku", "fecha_corte"]).sum())]}))
display(data[["fecha_corte", "sku", "ventas_7d", "disponible_fin", "objetivo_ventas_7d", "particion"]].head(12))
display(pd.DataFrame({"Permitidas": pd.Series(manifest["feature_columns"]),
                      "Excluidas": pd.Series(manifest["excluded_from_features"])}))

## 4. Separación temporal sin filtraciones

1. **Entrenamiento:** tres folds de 28 días con ajuste creciente. Dos configuraciones por modelo; gana el MAE medio de los folds.
2. **Selección:** primera parte de validación; compara cinco candidatos y dos baselines.
3. **Calibración:** últimos 28 días de validación; calcula intervalos para el ganador ya fijado.
4. **Prueba:** bloque reservado que se consulta después de registrar la elección. Se muestran todos los candidatos sin cambiar el ganador a posteriori.

Se purgan cortes cuyas etiquetas alcanzan el siguiente bloque. Todos los SKU comparten fechas. El preprocesamiento se ajusta dentro de cada fold y Gradient Boosting tiene desactivada la parada temprana aleatoria.

Se simula una predicción diaria: al cerrar el día siguiente se actualizan los lags con las observaciones recién conocidas. No se predice todo el periodo de prueba desde una única fecha.

In [ ]:
from ml.train import folds_for
training = data[data.particion.eq("entrenamiento")]
validation = data[data.particion.eq("validacion")]
calibration_start = validation.fecha_corte.max() - pd.Timedelta(days=27)
blocks = {"entrenamiento": training,
          "selección": validation[validation.fecha_objetivo_fin < calibration_start],
          "calibración": validation[validation.fecha_corte >= calibration_start],
          "prueba": data[data.particion.eq("prueba")]}
display(pd.DataFrame([{"Bloque": name, "Inicio": part.fecha_corte.min().date(),
                       "Fin corte": part.fecha_corte.max().date(), "Fin etiquetas": part.fecha_objetivo_fin.max().date(),
                       "Filas": len(part)} for name, part in blocks.items()]))
display(pd.DataFrame([{"Fold": n, "Última etiqueta ajuste": fit.fecha_objetivo_fin.max().date(),
                       "Evaluación desde": evaluation.fecha_corte.min().date(),
                       "Evaluación hasta": evaluation.fecha_corte.max().date()}
                     for n, fit, evaluation in folds_for(training, {**config, "folds": 1 if SMOKE else 3})]))

## 5. Métricas y regla de elección fijada antes de entrenar

| Métrica | Lectura |
|---|---|
| MAE | Error absoluto medio en unidades por SKU y siete días. Criterio principal. |
| RMSE | Da mayor peso a errores grandes. |
| WAPE | Error absoluto total / ventas observadas totales. N/D cuando no hay ventas. |
| Sesgo | Estimado − observado. Positivo: sobreestimación. |
| R² | Referencia adicional que puede ser negativa. |
| Cobertura | Porcentaje observado dentro del intervalo, comparado con 90% nominal. |

No se interpreta 1 − WAPE como exactitud. Gana el menor MAE de selección entre los cinco modelos. Para recomendar el candidato académico se exige mejora ≥2% frente al mejor baseline y sesgo absoluto ≤20%. Si falla, el informe conserva ese resultado y no fuerza una conclusión favorable.

## 6. Entrenamiento completo

La celda ejecuta búsqueda, selección, calibración, prueba y exportación. Guarda avances y ajusta modelos secuencialmente para limitar memoria. Ridge y Poisson usan categorías completas en formato disperso; los árboles agrupan categorías infrecuentes hasta 32 niveles por variable y usan las señales históricas de cada SKU.

Para controlar el costo de Colab, cada ajuste de búsqueda usa como máximo 80,000 filas del pasado de su fold, con semilla fija. Esto es una muestra dentro del entrenamiento, no una partición aleatoria de evaluación. Todos los candidatos comparten la misma muestra por fold; validación y prueba mantienen sus fechas y filas completas. Los cinco ajustes finales usan todo el entrenamiento. La búsqueda acotada puede perder configuraciones que funcionarían mejor con más datos.

El archivo final conserva el ajuste evaluado, sin reentrenar silenciosamente con la prueba. El código reutilizable está en ml/train.py, ml/reporting.py y ml/inference.py.

In [ ]:
del blocks, training, validation, data
import gc
gc.collect()
result = run_training(DATA, OUTPUT, smoke=SMOKE)
display(Markdown("**Elección:** " + result["selection"]["reason"]))
display(Markdown("**Estado:** " + result["selection"]["status"]))
print("Resultados:", OUTPUT)

## 7. Distribución y estabilidad temporal

La exploración usa entrenamiento. Los horizontes de siete días se solapan: no deben sumarse para calcular ventas de un mes. La estacionalidad es una hipótesis del simulador, no una medición del mercado peruano.

In [ ]:
def show_image(name):
    display(Image(filename=str(OUTPUT / "images" / name), width=1050))
show_image("01_distribucion.png")
show_image("02_validacion_temporal.png")
display(pd.read_csv(OUTPUT / "busqueda_temporal.csv"))

## 8. Modelos usados, ganador y justificación

Esta sección alimentará el dashboard: modelos, parámetros, tiempos, métricas y motivo de elección. El texto se calcula con los resultados de esta ejecución.

In [ ]:
selection = result["selection"]
display(pd.DataFrame(selection["comparisons"])[["model", "kind", "parameters", "cv_mae", "mae", "wape_pct", "bias_pct", "fit_seconds"]])
show_image("03_comparacion.png")
display(Markdown(selection["reason"]))
if not selection["selection_gate_passed"]:
    display(Markdown("**No supera el criterio frente al baseline.** Revisar datos y variables antes de desplegar."))
if SMOKE:
    display(Markdown("**Prueba técnica de 48 SKU.** Ejecutar modo completo para la comparación final del catálogo."))

## 9. Prueba reservada y matrices

La matriz de métricas compara regresiones. La matriz por rangos (0, 1–2, 3–5, 6–10, 11+) localiza subestimaciones y sobreestimaciones. Es un diagnóstico de cantidades agrupadas, no un nuevo clasificador ni un porcentaje de exactitud.

La correlación de Spearman se calcula sólo en entrenamiento y no demuestra causalidad.

In [ ]:
display(pd.DataFrame(result["evaluation"]["models"]))
show_image("04_matriz_metricas.png")
show_image("05_errores.png")
show_image("06_matriz_rangos.png")
show_image("07_correlaciones.png")

## 10. Intervalos y comparación con el baseline

Los radios se obtienen de errores absolutos de calibración, con cortes cada siete días y grupos de rotación reciente. La dependencia entre días y SKU impide garantizar formalmente cobertura de 90%; se informa la cobertura observada.

El intervalo de la diferencia de MAE usa bootstrap por bloques temporales de siete días. Si incluye cero, la evidencia de mejora es incierta incluso dentro de esta simulación.

In [ ]:
display(pd.DataFrame([result["evaluation"]["interval"]]))
display(pd.DataFrame([result["evaluation"]["paired_mae_difference"]]))
show_image("09_series.png")
show_image("10_cobertura.png")

## 11. Variables y segmentos con mayor error

La importancia por permutación mide el aumento de MAE al alterar cada variable en una muestra reproducible de prueba. Las variables correlacionadas pueden ocultar importancia. Es un diagnóstico y no se usa para cambiar el ganador después de ver la prueba.

Rotación semanal observada a partir de 28 días: baja <1, media <5 y alta ≥5 unidades. No utiliza objetivos futuros. El análisis por quiebre futuro sólo explica errores, no alimenta predicciones.

In [ ]:
show_image("08_importancia.png")
show_image("11_familias.png")
segments = pd.DataFrame(result["evaluation"]["segments"])
display(segments[segments.dimension.isin(["rotation", "stockout_diagnostic"])])
display(segments[segments.dimension.eq("categoria")].sort_values("mae", ascending=False).head(15))

## 12. Escenario de inventario

Se contrasta stock objetivo = techo(predicción) frente a ventas observadas de una semana sin solapamiento. Se muestran unidades sin cobertura y exceso de stock objetivo.

Es un escenario simplificado sin recepciones, plazos, reservas ni costos dentro del periodo. Las ventas están censuradas por el stock del simulador. **No calcula demanda perdida real, ahorro en soles ni órdenes automáticas.** La reposición posterior necesita evaluar esas condiciones con el kardex.

In [ ]:
scenario = json.loads((OUTPUT / "inventory_scenario.json").read_text(encoding="utf-8"))
display(pd.DataFrame(scenario["values"]))
show_image("12_inventario.png")

## 13. Cargar el modelo que utilizará la web

El paquete incorpora preprocesamiento y estimador, contrato de variables, versiones y hash. Cargar sólo artefactos propios de confianza: un hash no vuelve seguro un joblib de terceros.

La inferencia rechaza variables faltantes, valores no finitos, SKU desconocidos y stock incoherente. Debe habilitarse explícitamente el uso académico simulado. La web necesita registrar el mismo historial; exportar este modelo no demuestra que ya capture todas las variables operativas.

In [ ]:
from ml.inference import load_trusted_bundle, predict_bundle
bundle, contract = load_trusted_bundle(OUTPUT)
example = json.loads((OUTPUT / "inference_example.json").read_text(encoding="utf-8"))
display(predict_bundle(bundle, example["input"], allow_simulated=True))
print("Modelo:", contract["model_name"])
print("Fin del ajuste:", contract["training_end"])
print("Guardar/cargar y comparar predicciones:", result["roundtrip_prediction_check"])
try:
    predict_bundle(bundle, [{"sku": example["input"][0]["sku"]}], allow_simulated=True)
except ValueError as error:
    print("Validación esperada:", error)

## 14. Entrega al dashboard

| Artefacto | Uso |
|---|---|
| model.joblib + contract.json | Inferencia con contrato y versiones verificadas. |
| dashboard.json | Modelos, elección, razones, métricas, segmentos y limitaciones. |
| selection.json | Elección registrada antes de consultar prueba. |
| images/ | Doce gráficos en PNG y SVG. |
| JSON de gráficos y CSV de métricas | Gráficos interactivos sin extraer números de imágenes. |
| predicciones_prueba.csv.gz | Auditoría por fecha/SKU de todos los candidatos. |
| run_config.json, checksums.json, runtime-requirements.txt | Reproducibilidad, integridad y entorno. |

Conservar el ZIP completo. Los resultados de modo técnico deben mostrarse como evaluación reducida.

In [ ]:
archive = Path(str(OUTPUT) + "-web.zip")
assert archive.is_file()
display(pd.DataFrame([{"Archivo": str(p.relative_to(OUTPUT)), "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(OUTPUT.rglob("*")) if p.is_file()]))
print("ZIP:", archive, round(archive.stat().st_size / 1024**2, 2), "MB")
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))

## 15. Conclusión para la exposición

La conclusión usa las métricas obtenidas. Conserva tanto mejoras como fallos y explica el origen simulado de los datos.

In [ ]:
chosen_metrics = next(m for m in result["evaluation"]["models"] if m["selected"])
display(Markdown("**Modelo elegido:** " + selection["winner"] + ".\n\n" + selection["reason"]))
display(pd.DataFrame([chosen_metrics]))
display(Markdown("**Estado:** " + selection["status"] + ". La precisión real requiere evaluación con observaciones operativas."))
display(pd.DataFrame({"Limitaciones": result["limitations"]}))
print("Siguiente fase: recopilar variables operativas faltantes, conectar inferencia y mostrar la evaluación en el panel admin.")

## Referencias metodológicas

- [scikit-learn: variables rezagadas y evaluación temporal](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html)
- [Persistencia y compatibilidad de versiones](https://scikit-learn.org/stable/model_persistence.html)
- [Importancia por permutación](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [OneHotEncoder y categorías infrecuentes](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

Las tasas y estacionalidad de RODA son supuestos de data/ml_simulation_config.json, no mediciones externas.